# Detailed Investigation into Cotransmission
[fauai-12](notebooks/fauai-12_ViewCoTransmissionPatterns.ipynb) produced what seems to be a large number of co-transmitting neurons. I suspect these are false positives. Therefor the aim here is to go through the data carefully in order to investigate where the source of the false positives are.

In [1]:
#libraries
import os
import numpy as np
import pandas as pd
from decouple import config, Config, RepositoryEnv
from caveclient import CAVEclient
from fafbseg import flywire
from getpass import getpass
from sklearn.cluster import estimate_bandwidth, mean_shift
import seaborn as sns
import matplotlib.pyplot as plt
import navis
import datetime
import signal
import time

/opt/anaconda3/envs/fau_connectomics/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Check Client Connections

In [2]:
# Get variables from the .env file
ENV_PATH = "../.env"
config = Config(RepositoryEnv(ENV_PATH))

### CAV Client ###
# Get the CAVE_AUTH_TOKEN
cave_token = config("CAVE_AUTH_TOKEN", default=None)
if not cave_token:
    print("No CAVE token found in the .env file.")
    temp_token = getpass("Token not found in the .env file. Please enter your CAVE token (if you don't have one leave if blank and instructions will appear): ")
    if temp_token:
        cave_token = temp_token
    else:
        print("No CAVE token entered. Follow the information below to get a token.")
        CAVEclient.auth.get_new_token()
if cave_token:
    # Initialize the CAVE client
    client = CAVEclient('flywire_fafb_public')
    auth = client.auth
    tk = auth.token
    if not tk:
        print("Adding the token to your account. You only need to do this once.")
        auth.save_token(cave_token)
    else:
        print("CAVE token already exists in your account. No need to add it again.")

### Get the FLYWIRE_AUTH_TOKEN ###
flywire_token = config("FLYWIRE_AUTH_TOKEN", default=None)
tk = flywire.get_chunkedgraph_secret()
if not tk:
    print("No FLYWIRE token saved. Setting it with the value from the .env file.")
    if not flywire_token:
        flywire_token = getpass("Token not found in the .env file. Please enter your FLYWIRE token (if you don't have one leave if blank and instructions will appear): ")
        if not flywire_token:
            print("No FLYWIRE token entered. Follow the information below to get a token.")
            flywire.get_new_token()
        else:
            print("Setting the FLYWIRE token with the value from the .env file.")
            #Save the secret
            flywire.set_chunkedgraph_secret(flywire_token)
else:
    print("FLYWIRE token already exists in your account. No need to add it again.")

CAVE token already exists in your account. No need to add it again.
FLYWIRE token already exists in your account. No need to add it again.


## Adapt NT Metrics
Adjust NT_metrics from [fauai-11](notebooks/fauai-11_EvaluateAllVncCellTypes.ipynb)

### Single Neuron Codes

#### Load Codex Information

In [3]:
# Function 1: Get the dataframes
def get_codex_synapse_predictions(root_id, client):
    """
    Function to gather synapse predictions for a given root ID.
    """

    # Pull initial data from the CAVE client
    pre_df = client.materialize.query_table(
        table='synapses_nt_v1',
        filter_in_dict={'pre_pt_root_id': [root_id]},
    )

    return pre_df

# Function 2: Get the neuron skeleton
def get_neuron_skeleton(root_id):
    """
    Function to gather the neuron skeleton for a given root ID.
    """
    neuron_skeleton = flywire.get_skeletons(root_id)

    return neuron_skeleton

# Function 3: Identify NT contributions per neuron
def identify_nt_contributions(pre_df, softmax_threshold=0.25):
    """
    Function to identify neurotransmitter contributions per neuron.
    Inputs:
    pre_df: DataFrame containing synapse predictions with columns for each neurotransmitter type.
    softmax_threshold: Float, the threshold below which a neurotransmitter is considered 'Unknown'.

    Returns:
    nt_counts: a dictionary with counts of each neurotransmitter type.
    synapse_predictions: a numpy array with synapse predictions for each neurotransmitter type. (NB excludes 'Unknown' type)
    """

    #Synapse predictions
    synapse_predictions = np.zeros((len(pre_df), 6), dtype=np.float32)
    for i in range(pre_df.shape[0]):
        row = [pre_df.iloc[i]['gaba'], pre_df.iloc[i]['ach'], pre_df.iloc[i]['glut'], pre_df.iloc[i]['oct'], pre_df.iloc[i]['ser'], pre_df.iloc[i]['da']]
        synapse_predictions[i] = row
        
    # Define neurotransmitter types
    nt_types = ['GABA', 'ACh', 'Glut', 'Oct', 'Ser', 'DA', 'Unknown']
    
    # Initialize counts and filter along softmax values
    nt_counts = {nt: 0 for nt in nt_types}
    softmax_prediction_vals = {nt: [] for nt in nt_types}

    for i in range(pre_df.shape[0]):
        # Determine the synapse type based on the maximum prediction value  
        synapse_type = np.argmax(pre_df.iloc[i][['gaba', 'ach', 'glut', 'oct', 'ser', 'da']].values)
        softmax_val = np.max(pre_df.iloc[i][['gaba', 'ach', 'glut', 'oct', 'ser', 'da']].values)
        if softmax_val < softmax_threshold:
            synapse_type = 6  # Assign 'Unknown' if below threshold
        
        # Add the softmax value to the corresponding neurotransmitter type
        softmax_prediction_vals[nt_types[synapse_type]].append(softmax_val)
        
        # Increment the count for the neurotransmitter type
        nt_counts[nt_types[synapse_type]] += 1
    
    # Convert to ratios
    total_synapses = pre_df.shape[0]
    for nt in nt_counts.keys():
        if total_synapses > 0:
            nt_counts[nt] = nt_counts[nt] / total_synapses
        else:
            nt_counts[nt] = 0
    
    return nt_counts, synapse_predictions

##### Test on Example root_id

In [5]:
root_id="720575940627828552"
pre_df = get_codex_synapse_predictions(root_id, client)
print(f"pre_df:")
display(pre_df.head())
neuron_skeleton = get_neuron_skeleton(root_id)
print(f"neuron_skeleton: {neuron_skeleton}")
nt_counts, synapse_predictions = identify_nt_contributions(pre_df, softmax_threshold=0.25)
print(f"nt_counts: {nt_counts}")
print(f"synapse_predictions: {synapse_predictions}")

pre_df:


,id,created,superceded_id,valid,connection_score,cleft_score,gaba,ach,glut,oct,ser,da,valid_nt,pre_pt_supervoxel_id,pre_pt_root_id,post_pt_supervoxel_id,post_pt_root_id,pre_pt_position,post_pt_position
0,5058861,2021-03-09 20:14:58.183080+00:00,NaN,t,261.100586,150,0.001619,0.894702,0.070994,0.000157,0.000231,0.032296,t,79381166649734782,720575940627828552,79381166649741750,720575940620093019,"[510672, 337432, 169680]","[510584, 337496, 169680]"
1,75755908,2021-03-09 20:14:58.183080+00:00,NaN,t,9.543899,153,0.053037,0.673127,0.027152,0.000354,0.068373,0.177957,t,80295616592337085,720575940627828552,80295616592345840,720575940621440859,"[561748, 316328, 125640]","[561600, 316232, 125680]"
2,235642498,2021-03-09 20:14:58.183080+00:00,NaN,t,5.719914,8,0.012974,0.780764,0.005068,0.000644,0.008731,0.191818,t,80365779111493042,720575940627828552,80365779111501690,720575940621440859,"[568880, 303268, 120800]","[568948, 303284, 120880]"
3,212778560,2021-03-09 20:14:58.183080+00:00,NaN,t,231.785583,140,0.025942,0.867626,0.045978,0.001655,0.014550,0.044249,t,79873335407769768,720575940627828552,79873335407748645,720575940608029707,"[539704, 311604, 124400]","[539644, 311664, 124320]"
4,4486806,2021-03-09 20:14:58.183080+00:00,NaN,t,6.663127,0,0.014624,0.017974,0.007211,0.075861,0.645294,0.239036,t,80155428994104659,720575940627828552,80155428994100793,720575940632057570,"[554708, 346696, 169280]","[554684, 346628, 169240]"


neuron_skeleton: type                                             navis.TreeNeuron
name                                                     skeleton
id                                             720575940627828552
n_nodes                                                     12560
n_connectors                                                 None
n_branches                                                   1968
n_leafs                                                      2080
cable_length                                            6297333.0
soma            [3069, 7938, 7972, 7995, 8030, 8085, 8096, 125...
units                                                 1 nanometer
dtype: object
nt_counts: {'GABA': 0.016674962667994026, 'ACh': 0.8093578894972623, 'Glut': 0.03633648581383773, 'Oct': 0.004479840716774514, 'Ser': 0.08760577401692385, 'DA': 0.04479840716774515, 'Unknown': 0.0007466401194624191}
synapse_predictions: [[1.61904411e-03 8.94702494e-01 7.09942952e-02 1.56500872e-04
  2.311938

### Identification of NT cotransmission
Note I'm going to use NT_counts instead of the ratio calculation function to provide more detail.

RE the clustering function I suspect we will have to handle multiple bandwidths in the argument.

In [22]:
# Function 5: Mean Shift Clustering
def predict_neurotransmitter_usage_clustering(pre_df, synapse_predictions, bandwidth_quantile=1/7, min_synapses_ratio=0.01):
    """
    Function to predict neurotransmitter usage based on clustering.
    Returns a list of neurotransmitter types that are used based on clustering.

    Inputs:
    pre_df: DataFrame containing synapse predictions with columns for each neurotransmitter type.
    synapse_predictions: Numpy array with synapse predictions for each neurotransmitter type.
    bandwidth_quantile: Float, the quantile to use for bandwidth estimation.
    min_synapses_ratio: Float, the ratio of minimum synapses per cluster.
    
    Returns:
    nt_used: Dictionary of boolean values indicating whether each neurotransmitter is used.
    """
    temp_df = pre_df.copy()
    all_coords = np.array([pos for pos in pre_df['pre_pt_position'].values])
    total_synapses = all_coords.shape[0]

    # Calculate Parameters for mean shift clustering
    bandwidth = estimate_bandwidth(all_coords, quantile=bandwidth_quantile) #Set to 1/the total number of NT types (1/7)
    min_synapses = int(total_synapses * min_synapses_ratio)  # Set to 1% of the total number of synapses
    # print(f"Estimated bandwidth: {bandwidth}")
    # print(f"Minimum synapses per cluster: {min_synapses}")

    # Add the synapse type to the DataFrame
    nt_names = ['GABA', 'ACh', 'Glut', 'Oct', 'Ser', 'DA', 'Unknown']
    synapse_ids = []
    for i in range(pre_df.shape[0]):
        #Create a code for the synapse type
        synapse_type = np.argmax(synapse_predictions[i])
        softmax_val = np.max(synapse_predictions[i])
        if softmax_val < 0.25:
            synapse_type = 6
        synapse_ids.append(synapse_type)
    temp_df['synapse_type'] = synapse_ids
    nt_used = {}
    # Loop through each neurotransmitter type and apply mean shift clustering
    for nt_type in range(7):
        nt_df = temp_df[temp_df['synapse_type'] == nt_type]
        coords = np.array([pos for pos in nt_df['pre_pt_position'].values])
        if coords.shape[0] < 2:
            # print(f"Skipping NT type {nt_type} due to insufficient synapses.")
            continue
        cluster_centers, labels = mean_shift(coords, bandwidth=bandwidth)
        
        # Filter clusters with fewer than min_synapses
        unique_labels, counts = np.unique(labels, return_counts=True)
        valid_labels = unique_labels[counts >= min_synapses]        
        n_clusters = len(valid_labels)

        if n_clusters > 0:
            # print(f"NT type {nt_type} has a valid number of clusters:", n_clusters)
            nt_used[nt_names[nt_type]] = True
        else:
            # print(f"NT type {nt_type} has no valid clusters.")
            nt_used[nt_names[nt_type]] = False

    return nt_used

In [11]:
nt_counts

{'GABA': 0.016674962667994026,
 'ACh': 0.8093578894972623,
 'Glut': 0.03633648581383773,
 'Oct': 0.004479840716774514,
 'Ser': 0.08760577401692385,
 'DA': 0.04479840716774515,
 'Unknown': 0.0007466401194624191}

#### Test on example root id

In [10]:
nt_used = predict_neurotransmitter_usage_clustering(pre_df, synapse_predictions, bandwidth_quantile=1/100, min_synapses_ratio=0.01)
display(nt_used)

{0: False, 1: True, 2: False, 3: False, 4: False, 5: False, 6: False}

### Final neuron function

In [28]:
def identify_nt_contributions_for_neuron(root_id, 
                                         client, 
                                         softmax_thresholds=[0.25], 
                                         ratio_thresholds=[0.16], 
                                         bandwidth_quantile_values=[1/7], 
                                         min_synapses_ratio_values=[0.01],
                                         display_fig=False)-> dict:
    """
    Function to identify neurotransmitter contributions per neuron.
    Inputs:
    root_id: The root ID of the neuron to analyze.
    client: The CAVE client to use for querying data.
    softmax_thresholds: List, the thresholds below which a neurotransmitter is considered 'Unknown'.
    ratio_thresholds: List, the thresholds above which a neurotransmitter is considered used.
    bandwidth_quantile_values: List, the quantiles to use for bandwidth estimation in clustering.
    min_synapses_ratio_values: List, the ratios of minimum synapses required for a cluster to be considered valid.
    display_fig: Boolean, whether to display the figures or not.
    
    Returns a list of dictionaries with parameter settings.
    """
    # Get the synapse predictions
    pre_df = get_codex_synapse_predictions(root_id, client)
    neuron_results = []
    # Loop through parameters
    for softmax_threshold in softmax_thresholds:
        # Identify neurotransmitter contributions
        nt_counts, synapse_predictions = identify_nt_contributions(pre_df, softmax_threshold)

        # Now split up clustering values
        for bandwidth_quantile_value in bandwidth_quantile_values:
            for min_synapses_ratio_value in min_synapses_ratio_values:
                
                # Predict the neurotransmitter usage via clustering
                nt_clusters = predict_neurotransmitter_usage_clustering(pre_df,
                                                                    synapse_predictions, 
                                                                    bandwidth_quantile=bandwidth_quantile_value, 
                                                                    min_synapses_ratio=min_synapses_ratio_value)

                # Create a output dictionary to store the results
                nt_results = {
                    'root_id': root_id,
                    'softmax_threshold': softmax_threshold,
                    'bandwidth_quantile_value': bandwidth_quantile_value,
                    'min_synapses_ratio_value': min_synapses_ratio_value
                }

                # Add the nt_counts
                for key in nt_counts.keys():
                    new_key = f"{key}_ratio"
                    nt_results[new_key] = nt_counts[key]

                # Add the nt_clusters dictionary to the results with neurotransmitter names as key.
                for key in nt_clusters.keys():
                    new_key = f"{key}_clusters"
                    nt_results[new_key] = nt_clusters[key]

                # Append the results to the neuron_results list
                neuron_results.append(nt_results)

                if display_fig:
                    neuron_skeleton = get_neuron_skeleton(root_id)
                    # Plot the neuron skeleton with neurotransmitter contributions
                    fig, ax = plt.subplots(2,4, figsize=(20, 10))

                    # Plot the synapse predictions as a heatmap
                    sns.heatmap(synapse_predictions, ax=ax[0,0])
                    ax[0,0].set_title('Synapse Predictions Heatmap')
                    ax[0,0].set_xlabel('Neurotransmitter Type')
                    ax[0,0].set_xticks(np.arange(6) + 0.5)
                    ax[0,0].set_xticklabels(['GABA', 'ACh', 'Glut', 'Oct', 'Ser', 'DA'], rotation=45)

                    # Plot each row of the synapse predictions as a line plot
                    for i in range(synapse_predictions.shape[0]):
                        ax[0,1].plot(synapse_predictions[i], color='k', label=f'Synapse {i+1}', alpha=0.1)
                    mean_predictions = np.mean(synapse_predictions, axis=0)
                    median_predictions = np.median(synapse_predictions, axis=0)
                    ax[0,1].plot(mean_predictions, label='Mean', color='r')
                    ax[0,1].plot(median_predictions, label='Median', color='blue')
                    ax[0,1].set_xlabel('Neurotransmitter Type')
                    ax[0,1].set_xticks([0, 1, 2, 3, 4, 5])
                    ax[0,1].set_xticklabels(['GABA', 'ACh', 'Glut', 'Oct', 'Ser', 'DA'], rotation=45)

                    # Plot the neuron skeleton with the synapse predictions - all 3 combinations
                    navis.plot2d(neuron_skeleton, ax=ax[1,0], color='k', view='xy')
                    navis.plot2d(neuron_skeleton, ax=ax[1,1], color='k', view='yz')
                    navis.plot2d(neuron_skeleton, ax=ax[1,2], color='k', view='xz')

                    # Define neurotransmitter types and colors
                    present_types = set()
                    softmax_prediction_vals = {
                        'GABA': [],
                        'ACh': [],
                        'Glut': [],
                        'Oct': [],
                        'Ser': [],
                        'DA': [],
                        'Unknown': []
                    }
                    nt_types = ['GABA', 'ACh', 'Glut', 'Oct', 'Ser', 'DA', 'Unknown']
                    colours = ['red', 'green', 'blue', 'cyan', 'magenta', 'yellow', 'gray']

                    for i in range(synapse_predictions.shape[0]):
                        # Determine the synapse type based on the maximum prediction value  
                        synapse_type = np.argmax(synapse_predictions[i])
                        softmax_val = np.max(synapse_predictions[i])
                        if softmax_val < softmax_threshold:
                            synapse_type = 6  # Assign 'Unknown' if below threshold
                        colour = colours[synapse_type]
                        present_types.add(synapse_type)

                        # Add the softmax value to the corresponding neurotransmitter type
                        softmax_prediction_vals[nt_types[synapse_type]].append(softmax_val)
                    
                        # Find the position of the pre-synapse
                        pos = pre_df.iloc[i]['pre_pt_position']
                        pos_xy = (pos[0], pos[1])
                        pos_yz = (pos[1], pos[2])
                        pos_xz = (pos[0], pos[2])
                    
                        # Plot the synapse prediction on the neuron skeleton
                        ax[1,0].plot(pos_xy[0], pos_xy[1], 'o', color=colour, markersize=2, alpha=0.75)
                        ax[1,1].plot(pos_yz[0], pos_yz[1], 'o', color=colour, markersize=2, alpha=0.75)
                        ax[1,2].plot(pos_xz[0], pos_xz[1], 'o', color=colour, markersize=2, alpha=0.75)
                
                    # Create legend entries only for neurotransmitter types that are present
                    legend_elements = []
                    for nt_idx in sorted(present_types):
                        legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', 
                                                        markerfacecolor=colours[nt_idx], markersize=8,
                                                        label=nt_types[nt_idx]))
                
                    # Place the legend horizontally below the x-axis
                    ax[1,0].legend(
                        handles=legend_elements,
                        loc='upper center',
                        bbox_to_anchor=(0.5, -0.18),
                        borderaxespad=0.,
                        ncol=round(len(legend_elements)/2),
                        frameon=False
                    )
                    ax[1,0].set_title('Neuron Skeleton with Synapse Predictions - Coronal')
                    ax[1,1].set_title('Neuron Skeleton with Synapse Predictions - Sagittal')
                    ax[1,2].set_title('Neuron Skeleton with Synapse Predictions - Axial')

                    # Plot the counts of each neurotransmitter type
                    ax[0,3].bar(nt_counts.keys(), nt_counts.values(), color=colours)
                    ax[0,3].set_xlabel('Neurotransmitter Type')
                    ax[0,3].set_ylabel('Count')
                    ax[0,3].set_title('Counts of Neurotransmitter Types')

                    # Plot the counts of each neurotransmitter type
                    pcts = [nt_counts[nt] / synapse_predictions.shape[0] for nt in nt_counts.keys()]
                    ax[1,3].bar(nt_counts.keys(), pcts, color=colours)
                    ax[1,3].set_xlabel('Neurotransmitter Type')
                    ax[1,3].set_ylabel('Ratios')
                    ax[1,3].set_title('Ratios of Neurotransmitter Types')
                    
                    # Softmax confidence of the predictions
                    sns.stripplot(data=softmax_prediction_vals, ax=ax[0,2])
                    ax[0,2].set_title('Softmax Confidence of Predictions')
                    ax[0,2].set_xlabel('Neurotransmitter Type')
                    ax[0,2].set_ylabel('Softmax Confidence of chosen neurotransmitter')

                    plt.suptitle(f'Synapse Predictions for Neuron {root_id} with Softmax Threshold {softmax_threshold}', fontsize=16)
                    plt.tight_layout()
                    plt.show()
    
    return neuron_results

#### Example neuron test

In [23]:
root_id="720575940627828552"
result = identify_nt_contributions_for_neuron(root_id, client, softmax_thresholds=[0.25], bandwidth_quantile_values=[1/7, 1/25, 1/100], min_synapses_ratio_values=[0.01, 0.05, 0.1], display_fig=False)
display(pd.DataFrame(result))

Using bandwidth quantile: 0.14285714285714285, min synapses ratio: 0.01
Using bandwidth quantile: 0.14285714285714285, min synapses ratio: 0.05
Using bandwidth quantile: 0.14285714285714285, min synapses ratio: 0.1
Using bandwidth quantile: 0.04, min synapses ratio: 0.01
Using bandwidth quantile: 0.04, min synapses ratio: 0.05
Using bandwidth quantile: 0.04, min synapses ratio: 0.1
Using bandwidth quantile: 0.01, min synapses ratio: 0.01
Using bandwidth quantile: 0.01, min synapses ratio: 0.05
Using bandwidth quantile: 0.01, min synapses ratio: 0.1


,root_id,softmax_threshold,bandwidth_quantile_value,min_synapses_ratio_value,GABA_ratio,ACh_ratio,Glut_ratio,Oct_ratio,Ser_ratio,DA_ratio,Unknown_ratio,GABA_clusters,ACh_clusters,Glut_clusters,Oct_clusters,Ser_clusters,DA_clusters,Unknown_clusters
0,720575940627828552,0.25,0.142857,0.01,0.016675,0.809358,0.036336,0.00448,0.087606,0.044798,0.000747,False,True,False,False,True,True,False
1,720575940627828552,0.25,0.142857,0.05,0.016675,0.809358,0.036336,0.00448,0.087606,0.044798,0.000747,False,True,False,False,True,False,False
2,720575940627828552,0.25,0.142857,0.10,0.016675,0.809358,0.036336,0.00448,0.087606,0.044798,0.000747,False,True,False,False,False,False,False
3,720575940627828552,0.25,0.040000,0.01,0.016675,0.809358,0.036336,0.00448,0.087606,0.044798,0.000747,False,True,False,False,True,False,False
4,720575940627828552,0.25,0.040000,0.05,0.016675,0.809358,0.036336,0.00448,0.087606,0.044798,0.000747,False,True,False,False,False,False,False
5,720575940627828552,0.25,0.040000,0.10,0.016675,0.809358,0.036336,0.00448,0.087606,0.044798,0.000747,False,True,False,False,False,False,False
6,720575940627828552,0.25,0.010000,0.01,0.016675,0.809358,0.036336,0.00448,0.087606,0.044798,0.000747,False,True,False,False,False,False,False
7,720575940627828552,0.25,0.010000,0.05,0.016675,0.809358,0.036336,0.00448,0.087606,0.044798,0.000747,False,True,False,False,False,False,False
8,720575940627828552,0.25,0.010000,0.10,0.016675,0.809358,0.036336,0.00448,0.087606,0.044798,0.000747,False,False,False,False,False,False,False


## Find neurons of the appropriate cell type
There are three classification systems that I'm going to use. And we also need to get the metadata for the neuron that might be helpful

1. cell_type classification (Use regex expressions)
* Get all the DN neurons
* Get all the VUM neurons
* Get any neurons with AN (I don't think there are any)

2. cell_class classification
* If they are `AN`, `mechanosensory`, `unknown_sensory`, `gustatory`
--> NB These are big cell classes, will be hard to split up further
3. flow classification
* all `efferent` and `afferent` types

In [25]:
# Function to get the root IDs for a given cell type provided it meets the minimum synapse criteria
def get_roots_for_cell_types(client, expression, min_synapses=400)->pd.DataFrame:
    """
    Function to get the root IDs for a given cell type.
    Returns a datafrane of root IDs and associated metadata.

    Inputs:
    client: The CAVE client to use for querying data.
    expression: A regex expression to filter the cell types.
    min_synapses: Integer, the minimum number of synapses required for a root ID to be included in the results.
    """
    # Query the materialize table for the cell type
    roots_df = client.materialize.query_table(
        "hierarchical_neuron_annotations",
        filter_in_dict={'classification_system': ["cell_type"]},
        filter_regex_dict={'cell_type': expression}
    )

    # Now sort the DataFrame by cell_type and filter by num_synapses
    cell_types = roots_df['cell_type'].unique().tolist()
    n_cell_types = len(cell_types)
    print(f"Found {n_cell_types} unique cell types in the hierarchical_neuron_annotations table.")

    output_df = pd.DataFrame(columns=['root_id', 'classification_system', 'cell_type', 'num_synapses'])
    pct = 0
    for c, cell_type in enumerate(cell_types):
        # print(f"Cell type: {cell_type}, {np.round((((c+1)/n_cell_types)*100),2)}% complete")
        temp_df = roots_df[roots_df['cell_type'] == cell_type]
        pct_val = np.round((((c+1)/n_cell_types)*100))
        if pct_val > pct:
            print(f"{pct_val}% complete")
            pct = pct_val
        temp_root_ids = temp_df['pt_root_id'].tolist()
        n_neurons = len(temp_root_ids)
        # if n_neurons > 10 then we have concatenate to speed up the query
        if n_neurons <= 10:
            stats_df = client.materialize.query_table(
                table='synapses_nt_v1',
                filter_in_dict={'pre_pt_root_id': temp_root_ids}
                )
            # Now count the number of synapses per root
            check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

            root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
            num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

            final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
            final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

            type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
            type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

            output_df = pd.concat([output_df, type_df], ignore_index=True)
        else:
            # Loop through the root IDs in chunks of 10
            for i in range(0, n_neurons, 10):
                temp_root_ids_chunk = temp_root_ids[i:i+10]
                stats_df = client.materialize.query_table(
                    table='synapses_nt_v1',
                    filter_in_dict={'pre_pt_root_id': temp_root_ids_chunk}
                )
                # Now count the number of synapses per root
                check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

                root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
                num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

                final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
                final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

                type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
                type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

                output_df = pd.concat([output_df, type_df], ignore_index=True)
        c += 1

    return output_df

def get_roots_for_cell_class(client, specific_class, min_synapses=400)->dict:
    """
    Function to get the root IDs for a given cell type.
    Returns a dictionary of root IDs and the number of synapses.

    Inputs:
    client: The CAVE client to use for querying data.
    specific_class: The specific cell class to filter the root IDs by.
    min_synapses: Integer, the minimum number of synapses required for a root ID to be included in the results.
    """
    # Query the materialize table for the cell type
    roots_df = client.materialize.query_table(
        "hierarchical_neuron_annotations",
        filter_in_dict={'classification_system': ["cell_class"],
                        'cell_type': [specific_class]},
    )
    # Now sort the DataFrame by cell_type and filter by num_synapses
    cell_types = roots_df['cell_type'].unique().tolist()
    n_cell_types = len(cell_types)
    print(f"Found {n_cell_types} unique cell types in the hierarchical_neuron_annotations table.")
    
    pct = 0
    output_df = pd.DataFrame(columns=['root_id', 'classification_system', 'cell_type', 'num_synapses'])
    for c, cell_type in enumerate(cell_types):
        # print(f"Cell type: {cell_type}, {np.round((((c+1)/n_cell_types)*100),2)}% complete")
        temp_df = roots_df[roots_df['cell_type'] == cell_type]
        temp_root_ids = temp_df['pt_root_id'].tolist()
        n_neurons = len(temp_root_ids)

        pct_val = np.round((((c+1)/n_cell_types)*100),2)
        if pct_val > pct:
            print(f"{pct_val}% complete")
            pct = pct_val
        
        # if n_neurons > 10 then we have concatenate to speed up the query and avoid the CAVE timeout/synapse limit
        if n_neurons <= 10:
            stats_df = client.materialize.query_table(
                table='synapses_nt_v1',
                filter_in_dict={'pre_pt_root_id': temp_root_ids}
                )
            # Now count the number of synapses per root
            check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

            root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
            num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

            final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
            final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

            type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
            type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

            output_df = pd.concat([output_df, type_df], ignore_index=True)
        else:
            # Loop through the root IDs in chunks of 10
            print(f"Having to loop through the root IDs in chunks of 10 due to the number of neurons ({n_neurons}) being greater than 10.")
            for i in range(0, n_neurons, 10):
                # print(f"Evaluating chunk {i//10 + 1} of {n_neurons//10 + 1}.")
                temp_root_ids_chunk = temp_root_ids[i:i+10]
                stats_df = client.materialize.query_table(
                    table='synapses_nt_v1',
                    filter_in_dict={'pre_pt_root_id': temp_root_ids_chunk}
                )
                # Now count the number of synapses per root
                check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

                root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
                num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

                final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
                final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

                type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
                type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

                output_df = pd.concat([output_df, type_df], ignore_index=True)
        c += 1

    return output_df

def get_roots_for_cell_flow(client, specific_flow, min_synapses=400)->dict:
    """
    Function to get the root IDs for a given cell type.
    Returns a dictionary of root IDs and the number of synapses.

    Inputs:
    client: The CAVE client to use for querying data.
    specific_flow: The specific cell class to filter the root IDs by.
    min_synapses: Integer, the minimum number of synapses required for a root ID to be included in the results.
    """
    # Query the materialize table for the cell type
    roots_df = client.materialize.query_table(
        "hierarchical_neuron_annotations",
        filter_in_dict={'classification_system': ["flow"],
                        'cell_type': [specific_flow]},
    )
    # Now sort the DataFrame by cell_type and filter by num_synapses
    cell_types = roots_df['cell_type'].unique().tolist()
    n_cell_types = len(cell_types)
    print(f"Found {n_cell_types} unique cell types in the hierarchical_neuron_annotations table.")
    pct = 0
    output_df = pd.DataFrame(columns=['root_id', 'classification_system', 'cell_type', 'num_synapses'])

    for c, cell_type in enumerate(cell_types):
        # print(f"Cell type: {cell_type}, {np.round((((c+1)/n_cell_types)*100),2)}% complete")
        pct_val = np.round((((c+1)/n_cell_types)*100))
        if pct_val > pct:
            print(f"{pct_val}% complete")
            pct = pct_val

        temp_df = roots_df[roots_df['cell_type'] == cell_type]
        temp_root_ids = temp_df['pt_root_id'].tolist()
        n_neurons = len(temp_root_ids)
        print(f"There are {n_neurons} neurons for the cell type {cell_type}.")
        # if n_neurons > 10 then we have concatenate to speed up the query
        if n_neurons <= 10:
            stats_df = client.materialize.query_table(
                table='synapses_nt_v1',
                filter_in_dict={'pre_pt_root_id': temp_root_ids}
                )
            # Now count the number of synapses per root
            check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

            root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
            num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

            final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
            final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

            type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
            type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

            output_df = pd.concat([output_df, type_df], ignore_index=True)
        else:
            print(f"Having to loop through the root IDs in chunks of 10 due to the number of neurons ({n_neurons}) being greater than 10.")
            # Loop through the root IDs in chunks of 10
            for i in range(0, n_neurons, 10):
                # print(f"Chunk {i//10 + 1} of {n_neurons//10 + 1}.")
                temp_root_ids_chunk = temp_root_ids[i:i+10]
                stats_df = client.materialize.query_table(
                    table='synapses_nt_v1',
                    filter_in_dict={'pre_pt_root_id': temp_root_ids_chunk}
                )
                # Now count the number of synapses per root
                check_df = stats_df.groupby('pre_pt_root_id').size().reset_index(name='synapse_count')

                root_ids = check_df[check_df['synapse_count'] > min_synapses]['pre_pt_root_id'].tolist()
                num_synapses = check_df[check_df['synapse_count'] > min_synapses]['synapse_count'].tolist()

                final_df = roots_df[roots_df['pt_root_id'].isin(root_ids)].copy()
                final_df['num_synapses'] = final_df['pt_root_id'].map(dict(zip(root_ids, num_synapses)))

                type_df = final_df[['pt_root_id', 'classification_system', 'cell_type', 'num_synapses']].copy()
                type_df.rename(columns={'pt_root_id': 'root_id'}, inplace=True)

                output_df = pd.concat([output_df, type_df], ignore_index=True)
        c += 1

    return output_df

# Now loop and append the results for each neuron
def evaluate_all_neurons_for_classifier_cell_type(client, 
                                       classifier, 
                                       cell_type, 
                                       softmax_thresholds=[0.25], 
                                       ratio_thresholds=[0.16], 
                                       bandwidth_quantile_values=[1/7], 
                                       min_synapses_ratio_values=[0.01], 
                                       min_synapses_values=[400],
                                       neuron_ranges = [], 
                                       display_fig=False)-> list:
    """
    Function to evaluate all neurons for a given cell type.
    Inputs:
    client: The CAVE client to use for querying data.
    classifier: The classifier to use for the cell type. There are three options: 'cell_type', 'cell_class', 'flow'.
    cell_type: The cell type to evaluate.
    softmax_threshold: Float, the threshold below which a neurotransmitter is considered 'Unknown'.
    ratio_threshold: Float, the threshold above which a neurotransmitter is considered used.
    bandwidth_quantile_value: Float, the quantile to use for bandwidth estimation in clustering.
    min_synapses_ratio_value: Float, the ratio of minimum synapses required for a cluster to be considered valid.
    min_synapses: Integer, the minimum number of synapses required for a root ID to be included in the results.
    neuron_ranges: List of tuples, each containing the start and end indices for the neuron ranges to evaluate. e.g. [(0, 10), (10, 20)]
    display_fig: Boolean, whether to display the figure.


    Returns a tuple of two lists, containing dictionaries with neurotransmitter contributions for each neuron and root_ids which were timed out
    """
    # Check the classifier and get the root IDs
    min_synapses = np.min(min_synapses_values)
    if classifier == 'cell_type':
        print("Evaluating cell type classifier.")
        root_id_df = get_roots_for_cell_types(client, cell_type, min_synapses)
    elif classifier == 'cell_class':
        print("Evaluating cell class classifier.")
        root_id_df = get_roots_for_cell_class(client, cell_type, min_synapses)
    elif classifier == 'flow':
        print("Evaluating flow classifier.")
        root_id_df = get_roots_for_cell_flow(client, cell_type, min_synapses)
    else:
        raise ValueError(f"Unknown classifier: {classifier}")

    print(f"Found {len(root_id_df)} root IDs for cell type {cell_type} with at least {min_synapses} synapses.")
    nt_results = []
    # Return nothing if no root IDs found
    if root_id_df.empty:
        print("No root IDs found for the given cell type and classifier.")
        return nt_results
    total_neurons = len(root_id_df)
    # if neuron range is an empty list then we process all neurons
    if not neuron_ranges:
        for min_synapses in min_synapses_values: #filtering the root IDs by minimum synapses
            t_root_id = root_id_df[root_id_df['num_synapses'] >= min_synapses]
            for index, row in t_root_id.iterrows():
                print(f"\rProcessing {index+1} out of {total_neurons}", end="")
                root_id = row['root_id']
                num_synapses = row['num_synapses']

                # Identify neurotransmitter contributions for the neuron
                nt_result = identify_nt_contributions_for_neuron(root_id, 
                                                                client, 
                                                                softmax_thresholds, 
                                                                ratio_thresholds, 
                                                                bandwidth_quantile_values, 
                                                                min_synapses_ratio_values,
                                                                display_fig)
                # Create a dictionary to store the results and the metadata
                nt_result_df = pd.DataFrame(nt_result)
                n_combinations = len(nt_result)

                for i in range(n_combinations):
                    # Get the metadata for the root ID
                    metadata_dict = {
                        'root_id': root_id,
                        'classification_system': row['classification_system'],
                        'cell_type': row['cell_type'],
                        'num_synapses': num_synapses,
                        'min_synapses': min_synapses,
                    }
                    result_row = nt_result_df.iloc[i]
                    for key, value in result_row.items():
                        metadata_dict[key] = value

                    # for key, value in nt_result.items():
                    #     print(f"Adding {key}: {value[i]} to metadata_dict")
                    #     metadata_dict[key] = value[i] # Adding the i-th value for each key

                    # Append the result to the list
                    nt_results.append(metadata_dict)
    else:
        for neuron_range in neuron_ranges:
            start, end = neuron_range
            t_root_id = root_id_df[(root_id_df['num_synapses'] >= min_synapses) & (root_id_df.index >= start) & (root_id_df.index < end)]
            for index, row in t_root_id.iterrows():
                print(f"\rProcessing {index+1} out of {total_neurons}", end="")
                root_id = row['root_id']
                num_synapses = row['num_synapses']

                # Identify neurotransmitter contributions for the neuron
                nt_result = identify_nt_contributions_for_neuron(root_id, 
                                                                client, 
                                                                softmax_thresholds, 
                                                                ratio_thresholds, 
                                                                bandwidth_quantile_values, 
                                                                min_synapses_ratio_values,
                                                                display_fig)
                # Create a dictionary to store the results and the metadata
                nt_result_df = pd.DataFrame(nt_result)
                n_combinations = len(nt_result)

                for i in range(n_combinations):
                    # Get the metadata for the root ID
                    metadata_dict = {
                        'root_id': root_id,
                        'classification_system': row['classification_system'],
                        'cell_type': row['cell_type'],
                        'num_synapses': num_synapses,
                        'min_synapses': min_synapses,
                    }
                    result_row = nt_result_df.iloc[i]
                    for key, value in result_row.items():
                        metadata_dict[key] = value

                    # for key, value in nt_result.items():
                    #     print(f"Adding {key}: {value[i]} to metadata_dict")
                    #     metadata_dict[key] = value[i] # Adding the i-th value for each key

                    # Append the result to the list
                    nt_results.append(metadata_dict)

    return nt_results

## Main Investigation

### Test

In [30]:
classifier = "cell_type"
cell_types = ["DNg01_a", "DNg01_b", "DNg02_a"] # List of cell types to evaluate
eval_results = []
# Parameters
softmax_thresholds = [0.25]
bandwidth_quantile_values = [1/7, 1/25]
min_synapses_ratio_values = [0.01]
min_synapses_values = [400]

for cell_type in cell_types:
    nt_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                classifier, 
                                                cell_type, 
                                                softmax_thresholds=softmax_thresholds,
                                                bandwidth_quantile_values=bandwidth_quantile_values, 
                                                min_synapses_ratio_values=min_synapses_ratio_values,
                                                display_fig=False)
    eval_results.extend(nt_results)

# Convert the results to a DataFrame for better visualization
eval_df = pd.DataFrame(eval_results)

# Display the DataFrame
display(eval_df)

Evaluating cell type classifier.
Found 1 unique cell types in the hierarchical_neuron_annotations table.
100.0% complete
Found 1 root IDs for cell type DNg01_a with at least 400 synapses.
Processing 1 out of 1Evaluating cell type classifier.
Found 1 unique cell types in the hierarchical_neuron_annotations table.
100.0% complete
Found 1 root IDs for cell type DNg01_b with at least 400 synapses.
Processing 1 out of 1Evaluating cell type classifier.
Found 1 unique cell types in the hierarchical_neuron_annotations table.
100.0% complete
Found 1 root IDs for cell type DNg02_a with at least 400 synapses.
Processing 1 out of 1

,root_id,classification_system,cell_type,num_synapses,min_synapses,softmax_threshold,bandwidth_quantile_value,min_synapses_ratio_value,GABA_ratio,ACh_ratio,...,Ser_ratio,DA_ratio,Unknown_ratio,GABA_clusters,ACh_clusters,Glut_clusters,Ser_clusters,DA_clusters,Oct_clusters,Unknown_clusters
0,720575940622925972,cell_type,DNg01_a,421,400,0.25,0.142857,0.01,0.116390,0.584323,...,0.109264,0.047506,0.000000,True,True,True,True,True,NaN,NaN
1,720575940622925972,cell_type,DNg01_a,421,400,0.25,0.040000,0.01,0.116390,0.584323,...,0.109264,0.047506,0.000000,True,True,True,True,False,NaN,NaN
2,720575940628582607,cell_type,DNg01_b,596,400,0.25,0.142857,0.01,0.119128,0.572148,...,0.065436,0.107383,0.005034,True,True,True,True,True,False,False
3,720575940628582607,cell_type,DNg01_b,596,400,0.25,0.040000,0.01,0.119128,0.572148,...,0.065436,0.107383,0.005034,True,True,True,True,True,False,False
4,720575940610863310,cell_type,DNg02_a,617,400,0.25,0.142857,0.01,0.105348,0.716370,...,0.038898,0.079417,0.000000,True,True,True,True,True,False,NaN
5,720575940610863310,cell_type,DNg02_a,617,400,0.25,0.040000,0.01,0.105348,0.716370,...,0.038898,0.079417,0.000000,True,True,False,False,True,False,NaN


### Cell Type Searches

#### DNg types

In [ ]:
# Search by cell types
classifier = "cell_type"
#Potential thresholds
softmax_thresholds = [0.25]
bandwidth_quantile_values = [1/7, 1/25, 1/50, 1/100]
min_synapses_ratio_values = [0.01, 0.05, 0.1]
min_synapses_values = [400]

cell_types = [
    "DNg0.*",
    "DNg1.*",
    "DNg2.*",
    "DNg3.*",
    "DNg4.*",
    "DNg5.*",
    "DNg6.*",
    "DNg7.*",
    "DNg8.*",
    "DNg9.*",
]

for cell_type in cell_types:
    print(f"Evaluating cell type: {cell_type}")
    nt_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                classifier, 
                                                cell_type, 
                                                softmax_thresholds=softmax_thresholds, 
                                                bandwidth_quantile_values=bandwidth_quantile_values, 
                                                min_synapses_ratio_values=min_synapses_ratio_values,
                                                min_synapses_values=min_synapses_values,
                                                display_fig=False)
    # if nt_results dictionary is not empty
    nt_results_df = pd.DataFrame(nt_results)
    if not nt_results_df.empty:
        intermediary_csv_name = f"./new_nt_results_cell-type_{cell_type}.csv"
        print(f"Saving results to {intermediary_csv_name}")
        nt_results_df.to_csv(intermediary_csv_name, index=False)

Evaluating cell type: DNg0.*
Evaluating cell type classifier.
Found 20 unique cell types in the hierarchical_neuron_annotations table.
5.0% complete
10.0% complete
15.0% complete
20.0% complete
25.0% complete
30.0% complete
35.0% complete
40.0% complete
45.0% complete
50.0% complete
55.0% complete
60.0% complete
65.0% complete
70.0% complete
75.0% complete
80.0% complete
85.0% complete
90.0% complete
95.0% complete
100.0% complete
Found 61 root IDs for cell type DNg0.* with at least 400 synapses.
Processing 39 out of 61

#### DN[a-z]

In [ ]:
# Search by cell types
classifier = "cell_type"
#Potential thresholds
softmax_thresholds = [0.25]
bandwidth_quantile_values = [1/7, 1/25, 1/50, 1/100]
min_synapses_ratio_values = [0.01, 0.05, 0.1]
min_synapses_values = [400]

cell_types = [
    "DNa.*", 
    "DNb.*",
    "DNc.*",
    "DNd.*",
]
for cell_type in cell_types:
    print(f"Evaluating cell type: {cell_type}")
    nt_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                classifier, 
                                                cell_type, 
                                                softmax_thresholds=softmax_thresholds, 
                                                bandwidth_quantile_values=bandwidth_quantile_values, 
                                                min_synapses_ratio_values=min_synapses_ratio_values,
                                                min_synapses_values=min_synapses_values,
                                                display_fig=False)
    # if nt_results dictionary is not empty
    nt_results_df = pd.DataFrame(nt_results)
    if not nt_results_df.empty:
        intermediary_csv_name = f"./new_nt_results_cell-type_{cell_type}.csv"
        print(f"Saving results to {intermediary_csv_name}")
        nt_results_df.to_csv(intermediary_csv_name, index=False)

#### DNps[a-z]

In [ ]:
# Search by cell types
classifier = "cell_type"
#Potential thresholds
softmax_thresholds = [0.25]
bandwidth_quantile_values = [1/7, 1/25, 1/50, 1/100]
min_synapses_ratio_values = [0.01, 0.05, 0.1]
min_synapses_values = [400]


cell_types = [
    "DNp0.*",
    "DNp1.*",
    "DNp2.*",
    "DNp3.*",
    "DNp4.*",
    "DNp5.*",
    "DNp6.*",
    "DNp7.*",
    "DNp8.*",
    "DNp9.*",
]
for cell_type in cell_types:
    print(f"Evaluating cell type: {cell_type}")
    nt_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                classifier, 
                                                cell_type, 
                                                softmax_thresholds=softmax_thresholds, 
                                                bandwidth_quantile_values=bandwidth_quantile_values, 
                                                min_synapses_ratio_values=min_synapses_ratio_values,
                                                min_synapses_values=min_synapses_values,
                                                display_fig=False)
    # if nt_results dictionary is not empty
    nt_results_df = pd.DataFrame(nt_results)
    if not nt_results_df.empty:
        intermediary_csv_name = f"./new_nt_results_cell-type_{cell_type}.csv"
        print(f"Saving results to {intermediary_csv_name}")
        nt_results_df.to_csv(intermediary_csv_name, index=False)

#### Other Cell Types

In [ ]:
# Search by cell types
classifier = "cell_type"
#Potential thresholds
softmax_thresholds = [0.25]
bandwidth_quantile_values = [1/7, 1/25, 1/50, 1/100]
min_synapses_ratio_values = [0.01, 0.05, 0.1]
min_synapses_values = [400]


cell_types = [
    "DNx.*",
    ".*VUM.*",
    ".*AN.*"
]
for cell_type in cell_types:
    print(f"Evaluating cell type: {cell_type}")
    nt_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                classifier, 
                                                cell_type, 
                                                softmax_thresholds=softmax_thresholds, 
                                                bandwidth_quantile_values=bandwidth_quantile_values, 
                                                min_synapses_ratio_values=min_synapses_ratio_values,
                                                min_synapses_values=min_synapses_values,
                                                display_fig=False)
    # if nt_results dictionary is not empty
    nt_results_df = pd.DataFrame(nt_results)
    if not nt_results_df.empty:
        intermediary_csv_name = f"./new_nt_results_cell-type_{cell_type}.csv"
        print(f"Saving results to {intermediary_csv_name}")
        nt_results_df.to_csv(intermediary_csv_name, index=False)

### Cell Class Searches

In [ ]:
#Potential thresholds
softmax_thresholds = [0.25]
bandwidth_quantile_values = [1/7, 1/25, 1/50, 1/100]
min_synapses_ratio_values = [0.01, 0.05, 0.1]
min_synapses_values = [400]

#### AN Class

In [ ]:
# AN cell type
min_synapses = np.min(min_synapses_values)
an_root_id_df = get_roots_for_cell_class(client, "AN", min_synapses)
an_root_id_df.to_csv("./an_root_ids.csv", index=False)

print(f"Found {len(an_root_id_df)} root IDs for cell type AN with at least {min_synapses} synapses.")

In [ ]:
# load in the AN root_ids_df
an_root_id_df = pd.read_csv("./an_root_ids.csv")
nt_results = []
total_neurons = len(an_root_id_df)
cell_type = "AN"

# Now loop round the neuron ranges
neuron_ranges = [
    (0, 100),
    (100, 200),
    (200, 300),
    (300, 400),
    (400, 500),
    (500, 600),
    (600, 700),
    (700, 800),
    (800, 900),
    (900, 1000),
    (1000, 1100),
    (1100, 1200),
    (1200, 1300),
    (1300, 1400),
    (1400, 1500),
    (1500, 1600),
    (1600, 1700),
    (1700, 1800),
    (1800, 1900),
    (1900, 2000),
    (2000, 2100),
    (2100, 2200),
    (2200, 2300),
    (2300, 2400),
    (2400, 2500),
    (2500, 2600),
    (2600, 2700)
]
for cell_type in cell_types:
    print(f"Evaluating cell type: {cell_type}")
    nt_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                classifier, 
                                                cell_type, 
                                                softmax_thresholds=softmax_thresholds, 
                                                bandwidth_quantile_values=bandwidth_quantile_values, 
                                                min_synapses_ratio_values=min_synapses_ratio_values,
                                                min_synapses_values=min_synapses_values,
                                                display_fig=False)
    # if nt_results dictionary is not empty
    nt_results_df = pd.DataFrame(nt_results)
    if not nt_results_df.empty:
        intermediary_csv_name = f"./new_nt_results_cell-type_{cell_type}.csv"
        print(f"Saving results to {intermediary_csv_name}")
        nt_results_df.to_csv(intermediary_csv_name, index=False)

#### Mechanosensory Class

In [ ]:
# mechanosensory cell type
min_synapses = np.min(min_synapses_values)
mechanosensory_root_id_df = get_roots_for_cell_class(client, "mechanosensory", min_synapses)
mechanosensory_root_id_df.to_csv("./mechanosensory_root_ids.csv", index=False)

print(f"Found {len(mechanosensory_root_id_df)} root IDs for cell type `mechanosensory` with at least {min_synapses} synapses.")

In [ ]:
# load in the mechanosensory root_ids_df
mechanosensory_root_id_df = pd.read_csv("./mechanosensory_root_ids.csv")
nt_results = []
total_neurons = len(mechanosensory_root_id_df)
cell_type = "mechanosensory"

# Now loop round the neuron ranges
neuron_ranges = [
    (0, 100),
    (100, 200),
    (200, 300),
    (300, 400),
    (400, 500),
    (500, 600),
    (600, 700),
    (700, 800),
    (800, 900),
    # (900, 1000),
    # (1000, 1100),
    # (1100, 1200),
    # (1200, 1300),
    # (1300, 1400),
    # (1400, 1500),
    # (1500, 1600),
    # (1600, 1700),
    # (1700, 1800),
    # (1800, 1900),
    # (1900, 2000),
    # (2000, 2100),
    # (2100, 2200),
    # (2200, 2300),
    # (2300, 2400),
    # (2400, 2500),
    # (2500, 2600),
    # (2600, 2700)
]
for cell_type in cell_types:
    print(f"Evaluating cell type: {cell_type}")
    nt_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                classifier, 
                                                cell_type, 
                                                softmax_thresholds=softmax_thresholds, 
                                                bandwidth_quantile_values=bandwidth_quantile_values, 
                                                min_synapses_ratio_values=min_synapses_ratio_values,
                                                min_synapses_values=min_synapses_values,
                                                display_fig=False)
    # if nt_results dictionary is not empty
    nt_results_df = pd.DataFrame(nt_results)
    if not nt_results_df.empty:
        intermediary_csv_name = f"./new_nt_results_cell-type_{cell_type}.csv"
        print(f"Saving results to {intermediary_csv_name}")
        nt_results_df.to_csv(intermediary_csv_name, index=False)

#### unknown_sensory

In [ ]:
# unknown sensory cell type
min_synapses = np.min(min_synapses_values)
unknown_sensory_root_id_df = get_roots_for_cell_class(client, "unknown_sensory", min_synapses)
unknown_sensory_root_id_df.to_csv("./unknown_sensory_root_ids.csv", index=False)

print(f"Found {len(unknown_sensory_root_id_df)} root IDs for cell type `unknown_sensory` with at least {min_synapses} synapses.")

In [ ]:
# load in the unknown sensory root_ids_df
cell_type = "unknown_sensory"
unknown_sensory_root_id_df = pd.read_csv("./unknown_sensory_root_ids.csv")
nt_results = []
total_neurons = len(unknown_sensory_root_id_df)

# Now loop round the neuron ranges
neuron_ranges = [
    (0, 100),
    (100, 200),
    # (200, 300),
    # (300, 400),
    # (400, 500),
    # (500, 600),
    # (600, 700),
    # (700, 800),
    # (800, 900),
    # (900, 1000),
    # (1000, 1100),
    # (1100, 1200),
    # (1200, 1300),
    # (1300, 1400),
    # (1400, 1500),
    # (1500, 1600),
    # (1600, 1700),
    # (1700, 1800),
    # (1800, 1900),
    # (1900, 2000),
    # (2000, 2100),
    # (2100, 2200),
    # (2200, 2300),
    # (2300, 2400),
    # (2400, 2500),
    # (2500, 2600),
    # (2600, 2700)
]
for cell_type in cell_types:
    print(f"Evaluating cell type: {cell_type}")
    nt_results = evaluate_all_neurons_for_classifier_cell_type(client, 
                                                classifier, 
                                                cell_type, 
                                                softmax_thresholds=softmax_thresholds, 
                                                bandwidth_quantile_values=bandwidth_quantile_values, 
                                                min_synapses_ratio_values=min_synapses_ratio_values,
                                                min_synapses_values=min_synapses_values,
                                                display_fig=False)
    # if nt_results dictionary is not empty
    nt_results_df = pd.DataFrame(nt_results)
    if not nt_results_df.empty:
        intermediary_csv_name = f"./new_nt_results_cell-type_{cell_type}.csv"
        print(f"Saving results to {intermediary_csv_name}")
        nt_results_df.to_csv(intermediary_csv_name, index=False)